# YOLOv8s/m 追加実験 (Paper 1)

EXP-002 (YOLOv8n, mAP@0.5=0.58) と同じピラミッドaugmentedデータで、
YOLOv8s / YOLOv8m を学習し、モデルサイズによるmAP変化を確認する。

## 手順
1. Google Driveに `yolo_dataset.zip` をアップロード済みであること
2. ランタイム → ランタイムのタイプを変更 → **GPU (T4)** を選択
3. 上から順にセルを実行

## Step 1: セットアップ

In [ ]:
!pip install -q ultralytics
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

## Step 2: Google Driveマウント & データ展開

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# zipファイルのパスを確認してください
# デフォルト: マイドライブ直下に yolo_dataset.zip を置いた場合
ZIP_PATH = '/content/drive/MyDrive/yolo_dataset.zip'

!unzip -q {ZIP_PATH} -d /content/
!ls /content/data/processed/yolo_dataset/

In [ ]:
# dataset.yamlのパスをColab用に書き換え
import yaml

yaml_path = '/content/data/processed/yolo_dataset/dataset.yaml'
with open(yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['path'] = '/content/data/processed/yolo_dataset'

with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f)

print('Updated dataset.yaml:')
!cat {yaml_path}

## Step 3: YOLOv8s 学習 (30 epochs)

In [ ]:
from ultralytics import YOLO

model_s = YOLO('yolov8s.pt')
results_s = model_s.train(
    data='/content/data/processed/yolo_dataset/dataset.yaml',
    epochs=30,
    batch=16,
    imgsz=640,
    device=0,
    workers=2,
    seed=0,
    deterministic=True,
    project='/content/results',
    name='pyramid_yolov8s',
    exist_ok=True,
)
print('YOLOv8s training complete!')

## Step 4: YOLOv8m 学習 (30 epochs)

In [ ]:
model_m = YOLO('yolov8m.pt')
results_m = model_m.train(
    data='/content/data/processed/yolo_dataset/dataset.yaml',
    epochs=30,
    batch=16,
    imgsz=640,
    device=0,
    workers=2,
    seed=0,
    deterministic=True,
    project='/content/results',
    name='pyramid_yolov8m',
    exist_ok=True,
)
print('YOLOv8m training complete!')

## Step 5: 結果確認

In [ ]:
import pandas as pd

print('=== YOLOv8n (EXP-002, reference) ===')
print('mAP@0.5 = 0.58, mAP@0.5:0.95 = 0.314')
print()

for name in ['pyramid_yolov8s', 'pyramid_yolov8m']:
    csv_path = f'/content/results/{name}/results.csv'
    try:
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        best_map50 = df['metrics/mAP50(B)'].max()
        best_map5095 = df['metrics/mAP50-95(B)'].max()
        best_epoch = df['metrics/mAP50(B)'].idxmax() + 1
        print(f'=== {name} ===')
        print(f'Best mAP@0.5:     {best_map50:.4f} (epoch {best_epoch})')
        print(f'Best mAP@0.5:0.95: {best_map5095:.4f}')
        print(f'Improvement over nano: {(best_map50 - 0.58) / 0.58 * 100:+.1f}%')
        print()
    except Exception as e:
        print(f'{name}: {e}')

## Step 6: 結果をDriveに保存

In [ ]:
# 結果をzipにしてDriveに保存
!zip -r /content/drive/MyDrive/yolov8_results.zip /content/results/
print('Results saved to Google Drive: yolov8_results.zip')
print('このファイルをダウンロードしてClaude Codeに渡してください')